In [0]:
/*
=============================================================================
PURPOSE: Market Beta Dimension Table for Risk Analysis
=============================================================================
This view provides a standardized reference table of beta coefficients used
for portfolio risk assessment and asset classification.

TARGET SCHEMA: thekitchen.t (transformed data layer)
GRAIN: One row per beta category

WHAT IS BETA?
Beta (β) measures an asset's volatility relative to the overall market:
- β = 1.00 → Asset moves exactly with the market (100% correlation)
- β < 1.00 → Defensive/Low-risk (less volatile than market)
- β > 1.00 → Aggressive/High-risk (more volatile than market)
- β = 0.50 → Asset moves 50% as much as the market
- β = 1.50 → Asset moves 150% as much as the market

BUSINESS VALUE:
- Enables portfolio risk categorization and analysis
- Supports CAPM (Capital Asset Pricing Model) calculations
- Facilitates what-if scenarios with different risk profiles
- Provides consistent risk labels for reporting and visualization

USAGE PATTERN:
- Join to portfolio holdings or securities on BetaKey
- Use in scenario analysis: "What if my portfolio had β = 1.3?"
- Filter by risk appetite: defensive (β < 1.0) vs growth (β > 1.0)
- Calculate expected returns: E(R) = Risk-Free Rate + β × Market Risk Premium
=============================================================================
*/

CREATE OR REPLACE VIEW t.marketbeta_tmp1 AS
-- ============================================
-- Beta Categories Reference Data
-- ============================================
-- Inline table of standardized beta values for risk modeling
SELECT
  *
FROM
  VALUES
    -- DEFENSIVE STRATEGIES (β < 1.0)
    -- Lower volatility, capital preservation focus
    ('BETA_050', CAST(0.50 AS DECIMAL(6, 3)), '0.50 - Very Defensive'),     -- Utilities, consumer staples
    ('BETA_070', CAST(0.70 AS DECIMAL(6, 3)), '0.70 - Defensive'),          -- Healthcare, telecom
    ('BETA_085', CAST(0.85 AS DECIMAL(6, 3)), '0.85 - Low Market Risk'),    -- Large-cap value stocks
    
    -- MARKET BENCHMARK (β = 1.0)
    -- Market-level risk, index fund behavior
    ('BETA_100', CAST(1.00 AS DECIMAL(6, 3)), '1.00 - Market'),             -- S&P 500, broad market index
    
    -- GROWTH STRATEGIES (β > 1.0)
    -- Higher volatility, capital appreciation focus
    ('BETA_115', CAST(1.15 AS DECIMAL(6, 3)), '1.15 - Moderate Growth'),    -- Growth-oriented large caps
    ('BETA_130', CAST(1.30 AS DECIMAL(6, 3)), '1.30 - Growth / Tech'),      -- Tech sector, innovation stocks
    ('BETA_150', CAST(1.50 AS DECIMAL(6, 3)), '1.50 - High Risk')           -- Small caps, emerging tech
    
  AS v (BetaKey, BetaValue, BetaLabel);  -- Column aliases for the VALUES clause

/*
=============================================================================
USAGE EXAMPLES:
=============================================================================

-- Example 1: Calculate expected returns using CAPM
-- E(R) = Risk-Free Rate + Beta × Market Risk Premium
SELECT
  b.BetaLabel,
  b.BetaValue,
  rf.Yield AS risk_free_rate,
  mrp.MrpValue AS market_risk_premium,
  (rf.Yield + (b.BetaValue * mrp.MrpValue)) AS expected_return,
  ROUND((rf.Yield + (b.BetaValue * mrp.MrpValue)) * 100, 2) AS expected_return_pct
FROM t.marketbeta_tmp1 b
CROSS JOIN (
  SELECT Yield FROM t.us_treasury_ten_years 
  WHERE Date = (SELECT MAX(Date) FROM t.us_treasury_ten_years)
  LIMIT 1
) rf
CROSS JOIN (
  SELECT MrpValue FROM t.marketriskpremium_tmp1 WHERE MrpScenario = 'Base' LIMIT 1
) mrp
ORDER BY b.BetaValue;

-- Example 2: What-if scenario analysis
-- Compare different portfolio beta assumptions
WITH scenario AS (
  SELECT 'Current Portfolio' AS scenario, 1.15 AS portfolio_beta
  UNION ALL
  SELECT 'Conservative Shift', 0.85
  UNION ALL
  SELECT 'Growth Tilt', 1.30
)
SELECT
  s.scenario,
  s.portfolio_beta,
  b.BetaLabel AS closest_category,
  ABS(s.portfolio_beta - b.BetaValue) AS beta_difference
FROM scenario s
CROSS JOIN t.marketbeta_tmp1 b
QUALIFY ROW_NUMBER() OVER (PARTITION BY s.scenario ORDER BY ABS(s.portfolio_beta - b.BetaValue)) = 1;

-- Example 3: Sensitivity analysis across market risk premium scenarios
-- Shows how expected returns vary with different MRP assumptions
SELECT
  b.BetaLabel,
  m.MrpScenario,
  rf.Yield AS risk_free_rate,
  m.MrpValue AS market_risk_premium,
  ROUND((rf.Yield + (b.BetaValue * m.MrpValue)) * 100, 2) AS expected_return_pct
FROM t.marketbeta_tmp1 b
CROSS JOIN (
  SELECT Yield FROM t.us_treasury_ten_years 
  WHERE Date = (SELECT MAX(Date) FROM t.us_treasury_ten_years)
) rf
CROSS JOIN t.marketriskpremium_tmp1 m
WHERE m.MrpScenario IN ('Low', 'Base', 'High')
ORDER BY b.BetaValue, m.SortOrder;
=============================================================================
*/